# Chapitre 0.4 : les tenseurs

Une IA ne manipule que des nombres, rangés dans des tableaux : des **tenseurs**. Dans ce carnet, tu crées des tenseurs avec PyTorch, tu les lis, tu calcules avec, et tu apprends à prédire la forme d’un résultat avant de lancer le calcul. Pour finir, tu chronomètres le même calcul sur le processeur, et sur la carte graphique si elle est là.

Ce carnet accompagne [la page du chapitre 0.4](https://chipat-neko.github.io/forge-ia/partie-0/0-4/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. créer des tenseurs de 0 à 3 dimensions ;
2. lire leur forme, leur type, leur appareil et leurs nombres ;
3. calculer case par case, puis en produit matriciel ;
4. comprendre la règle de la diffusion, qui dit quelle forme aura `a + b` ;
5. comprendre celle du produit matriciel, `a @ b` ;
6. compter la mémoire d’un tenseur ;
7. chronométrer un produit de matrices, sur le processeur et sur la carte ;
8. relever un défi, puis vérifier que ce carnet suit les mêmes règles que la page du site.

**Comment l’utiliser**

Il n’y a rien à installer : Colab a déjà Python et PyTorch, et ce carnet ne les installe pas. Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Change les nombres des cellules et relance-les : c’est fait pour.

**Pour avoir une carte graphique dans Colab** : menu « Exécution », puis « Modifier le type d’exécution ». Choisis un accélérateur « GPU », puis enregistre. Colab redémarre : relance les cellules depuis le début. L’offre gratuite ne garantit pas de carte, et une session dure au plus 12 heures. Sans carte, tout marche aussi, sur le processeur.

## Étape 0 : récupérer le code

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, puis indique à Python où trouver le projet. Le projet utilise PyTorch, que Colab a déjà installé.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia

projet = depot / "projets" / "0.4-tenseurs"
if not projet.exists():
    raise FileNotFoundError("Le dépôt n’a pas pu être téléchargé. Vérifie ta connexion, puis relance cette cellule.")

# Python cherche les modules dans les dossiers de sys.path : on y ajoute le projet.
sys.path.insert(0, str(projet.resolve()))
print("Le code du projet est prêt.")

## Étape 1 : créer des tenseurs

`torch.tensor` fabrique un tenseur à partir d’une liste de nombres, `torch.zeros` le remplit de zéros, et `torch.arange` compte à partir de 0. `reshape` range ensuite les nombres en lignes et en colonnes.

La **forme** d’un tenseur dit combien il a de nombres dans chaque direction :

- `()` pour un nombre seul ;
- `(4,)` pour un vecteur de 4 nombres : la virgule est celle de Python, qui écrit ainsi un tuple d’un seul nombre ;
- `(2, 3)` pour une matrice de 2 lignes et 3 colonnes ;
- `(2, 3, 4)` pour deux matrices de 3 lignes et 4 colonnes, l’une derrière l’autre.

Le nombre de **dimensions** est la longueur de la forme : de 0 à 3 ici. Le **type** dit comment chaque nombre est rangé : `torch.float32` pour un nombre à virgule de 32 bits, le type des IA, et `torch.int64` pour un entier.

In [ ]:
import torch

nombre = torch.tensor(3.5)  # un nombre seul
vecteur = torch.tensor([1.0, 2.0, 3.0, 4.0])  # une liste de nombres
matrice = torch.arange(6.0).reshape(2, 3)  # 0 à 5, rangés en 2 lignes de 3
lot = torch.zeros(2, 3, 4)  # deux matrices de 3 × 4 zéros
entiers = torch.tensor([1, 2, 3])  # sans virgule, ce sont des entiers

print(f"PyTorch {torch.__version__}")
for nom, t in [("nombre", nombre), ("vecteur", vecteur), ("matrice", matrice), ("lot", lot), ("entiers", entiers)]:
    print(f"{nom}\u202f: forme {tuple(t.shape)}, {t.dim()} dimension(s), type {t.dtype}")
print()
print(matrice)

## Étape 2 : lire un tenseur

Un tenseur sait tout de lui-même : sa forme (`shape`), son nombre de dimensions (`dim()`), son nombre de nombres (`numel()`), son type (`dtype`) et son **appareil** (`device`), la mémoire où il est rangé : `cpu` pour celle du processeur, `cuda` pour celle de la carte graphique.

Pour lire ses nombres, on donne leur place entre crochets, en comptant à partir de 0 : `m[1, 2]` est la case de la ligne 1 et de la colonne 2. `m[1]` est toute la ligne 1, et `m[:, 0]` toute la colonne 0 : le `:` veut dire « toutes les lignes ». `item()` sort un nombre seul du tenseur, et `tolist()` en sort une liste de Python.

In [ ]:
m = torch.arange(6.0).reshape(2, 3)
print(m)
print(f"forme\u202f: {tuple(m.shape)}")
print(f"dimensions\u202f: {m.dim()}")
print(f"nombres\u202f: {m.numel()}")
print(f"type\u202f: {m.dtype}")
print(f"appareil\u202f: {m.device}")
print(f"case\u202f: {m[1, 2].item()}")  # ligne 1, colonne 2
print(f"ligne\u202f: {m[1].tolist()}")
print(f"colonne\u202f: {m[:, 0].tolist()}")

## Étape 3 : calculer

`a + b` et `a * b` calculent **case par case** : chaque case de `a` avec la case de `b` à la même place. `a @ b` est le **produit matriciel** de 0.2 : chaque ligne de `a` avec chaque colonne de `b`, en produit scalaire. C’est le calcul qui domine l’entraînement d’une IA.

`sum()` additionne tous les nombres, et `mean(dim=0)` fait la moyenne en descendant, colonne par colonne. La dernière ligne ajoute un vecteur `(2,)` à une matrice `(2, 2)` : PyTorch l’ajoute à chaque ligne. C’est la **diffusion**, l’étape suivante.

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[10.0, 20.0], [30.0, 40.0]])
print(f"a + b\u202f: {(a + b).tolist()}")
print(f"a * b\u202f: {(a * b).tolist()}")
print(f"a @ b\u202f: {(a @ b).tolist()}")
print(f"a.sum()\u202f: {a.sum().item()}")
print(f"a.mean(dim=0)\u202f: {a.mean(dim=0).tolist()}")
print(f"a + (100, 200)\u202f: {(a + torch.tensor([100.0, 200.0])).tolist()}")

## Étape 4 : la règle de la diffusion

Pour calculer `a + b` quand les formes diffèrent, PyTorch **aligne les deux formes par la droite**, puis les compare dimension par dimension :

- deux tailles égales restent telles quelles ;
- une taille 1 s’étire pour suivre l’autre : la même valeur est reprise sur toute la longueur ;
- une dimension qui manque, à gauche, compte pour 1 ;
- deux tailles différentes, sans 1, rendent le calcul impossible.

Ainsi, `(3, 1) + (4,)` donne `(3, 4)` : la colonne de 3 nombres s’étire sur 4 colonnes, et la ligne de 4 nombres sur 3 lignes.

`forme_diffusee(a, b)`, dans `tenseurs/formes.py`, applique cette règle en Python pur, sans PyTorch. La cellule la compare à `torch.broadcast_shapes`, qui donne la forme que prendrait PyTorch, puis fait le vrai calcul de `(3, 1) + (4,)`.

In [ ]:
from tenseurs.formes import forme_diffusee, forme_produit


def ecrire(forme):
    return "impossible" if forme is None else str(forme)


def diffusion_de_pytorch(a, b):
    try:
        return tuple(torch.broadcast_shapes(a, b))
    except RuntimeError:
        return None


for a, b in [((3, 1), (4,)), ((2, 3), (3,)), ((2, 3), (2,)), ((2, 3), (3, 2))]:
    print(f"{a} + {b}\u202f: {ecrire(forme_diffusee(a, b))} selon la règle, {ecrire(diffusion_de_pytorch(a, b))} selon PyTorch.")

# Le vrai calcul : une colonne (3, 1) plus une ligne (4,).
colonne = torch.tensor([[0.0], [10.0], [20.0]])
ligne = torch.tensor([1.0, 2.0, 3.0, 4.0])
print()
print(colonne + ligne)

## Étape 5 : la règle du produit matriciel

`a @ b` multiplie chaque ligne de `a` par chaque colonne de `b` : il faut autant de colonnes dans `a` que de lignes dans `b`. Une matrice `(2, 3)` fois une matrice `(3, 4)` donne une matrice `(2, 4)`.

`torch.matmul`, qu’on écrit `@`, va plus loin :

- un vecteur à droite se lit comme une colonne : `(2, 3) @ (3,)` donne `(2,)` ;
- un vecteur à gauche se lit comme une ligne : `(3,) @ (3, 4)` donne `(4,)` ;
- deux vecteurs donnent un nombre seul, leur produit scalaire : `(3,) @ (3,)` donne `()` ;
- au-delà de deux dimensions, les dernières sont les matrices, et les premières, le **lot**, se diffusent comme pour `a + b` : `(5, 2, 3) @ (3, 4)` fait 5 produits, et donne `(5, 2, 4)`.

`forme_produit(a, b)` applique ces règles en Python pur. La cellule la compare à la forme que donne `torch.matmul` sur des tenseurs vides de ces formes.

In [ ]:
def produit_de_pytorch(a, b):
    try:
        return tuple(torch.matmul(torch.empty(a), torch.empty(b)).shape)
    except RuntimeError:
        return None


for a, b in [((2, 3), (3, 4)), ((2, 3), (3,)), ((3,), (3,)), ((5, 2, 3), (3, 4)), ((2, 3), (2, 3))]:
    print(f"{a} @ {b}\u202f: {ecrire(forme_produit(a, b))} selon la règle, {ecrire(produit_de_pytorch(a, b))} selon PyTorch.")

## Étape 6 : la mémoire d’un tenseur

Un tenseur range ses nombres les uns à la suite des autres. Sa mémoire, c’est le nombre de nombres, fois la place d’un nombre : 4 octets en `float32`, 2 en `bfloat16`, 8 en `float64`. `element_size()` donne cette place.

`octets(forme, dtype)`, dans `tenseurs/memoire.py`, fait ce calcul sans créer le tenseur. Ici, un Mo vaut 1 024 × 1 024 = 1 048 576 octets, comme dans l’explorateur de fichiers de Windows.

In [ ]:
from tenseurs.chrono import en_francais
from tenseurs.memoire import octets

MO = 1024 * 1024
for forme, dtype in [((4096, 4096), torch.float32), ((4096, 4096), torch.bfloat16), ((4096, 4096), torch.float64)]:
    n = octets(forme, dtype)
    print(f"{forme} en {dtype}\u202f: {en_francais(n)} octets, soit {en_francais(n / MO)} Mo")

# Le même compte, sur un vrai tenseur.
t = torch.zeros(4096, 4096)
print(f"torch.zeros(4096, 4096)\u202f: {en_francais(t.numel())} nombres de {t.element_size()} octets")

## Étape 7 : chronométrer un produit de matrices

Le même produit de deux matrices, en `float32`, chronométré sur trois tailles : 256 × 256, 1 024 × 1 024 et 2 048 × 2 048. Quand la taille double, le calcul demande 8 fois plus de multiplications : 2 × 2 × 2, pour les lignes, les colonnes, et la longueur de chaque produit scalaire.

`chronometrer_produit(taille, appareil)`, dans `tenseurs/chrono.py`, fait deux tours pour rien, qui préparent le calcul, puis en garde la durée du milieu sur cinq, en millisecondes. Sur la carte, elle attend que le calcul soit vraiment fini, avec `torch.cuda.synchronize()`. La carte n’est chronométrée que si PyTorch en voit une.

In [ ]:
from tenseurs.chrono import chronometrer_produit

sur_la_carte = torch.cuda.is_available()
print(f"Le processeur a {os.cpu_count()} fils, et PyTorch en utilise {torch.get_num_threads()}.")
for taille in (256, 1024, 2048):
    ligne = f"{en_francais(taille)} × {en_francais(taille)}\u202f: processeur {en_francais(chronometrer_produit(taille, 'cpu'))} ms"
    if sur_la_carte:
        ligne += f", carte {en_francais(chronometrer_produit(taille, 'cuda'))} ms"
    print(ligne)
if not sur_la_carte:
    print("Pas de carte graphique ici\u202f: seul le processeur est chronométré.")
    print("Dans Colab, pour avoir une carte\u202f: Exécution > Modifier le type d’exécution > GPU, puis relance le carnet depuis le début.")

## Exercice : prédire une forme

Avant de lancer un calcul, un bon réflexe est de prédire la forme du résultat : une erreur de forme est l’erreur la plus courante quand on écrit une IA.

Que donne `(2, 1, 3) + (4, 1)` ? Écris ta prédiction à la place de `()`, sous la forme d’un tuple comme `(5, 6)`, puis lance la cellule. Elle fait le calcul avec PyTorch, et compare.

Indice : aligne les deux formes par la droite, comme dans l’étape 4, puis compare-les dimension par dimension.

In [ ]:
# À toi : remplace () par la forme que tu prédis pour (2, 1, 3) + (4, 1).
ma_prediction = ()

# Ne change rien en dessous : cette partie fait le calcul avec PyTorch, et compare.
vraie = tuple((torch.zeros(2, 1, 3) + torch.zeros(4, 1)).shape)
if tuple(ma_prediction) == vraie:
    print(f"Défi réussi\u202f: (2, 1, 3) + (4, 1) donne bien {vraie}.")
    print(f"La règle du projet est d’accord\u202f: forme_diffusee((2, 1, 3), (4, 1)) donne {forme_diffusee((2, 1, 3), (4, 1))}.")
else:
    print(f"Pas encore. Ta prédiction est {tuple(ma_prediction)}. Aligne les deux formes par la droite, puis compare-les dimension par dimension.")

<details>
<summary>Voir une correction (essaie d’abord !)</summary>

Alignées par la droite, les deux formes se comparent de droite à gauche :

- la dernière dimension : 3 et 1. Le 1 s’étire : 3 ;
- l’avant-dernière : 1 et 4. Le 1 s’étire : 4 ;
- la première : 2, et rien en face, qui compte pour 1 : 2.

Le résultat a la forme `(2, 4, 3)`. Remplace la première ligne de code de la cellule de l’exercice par :

```python
ma_prediction = (2, 4, 3)
```

</details>

## Dernière étape : les mêmes règles que la page ?

La page du site ne lance pas PyTorch : son atelier des formes applique les deux règles en TypeScript, et elles sont vérifiées sur un fichier de référence, `tests/attendus-formes.json`. Ce fichier a été écrit avec PyTorch, pour toutes les formes de 1 à 3 dimensions de tailles 1 à 3, prises deux à deux : 1 521 paires.

La cellule suivante redemande ces formes au PyTorch de ce carnet, les compare au fichier et aux règles du projet. Puis elle dit sur quoi ce carnet a tourné.

In [ ]:
import json

from tenseurs.attendus import calculer_paires
from tenseurs.formes import forme_diffusee, forme_produit


def en_liste(forme):
    return None if forme is None else list(forme)


fichier = json.loads((projet / "tests" / "attendus-formes.json").read_text(encoding="utf-8"))
paires = calculer_paires()  # les formes que donne le PyTorch de ce carnet
regles = all(
    p["diffusion"] == en_liste(forme_diffusee(tuple(p["a"]), tuple(p["b"])))
    and p["produit"] == en_liste(forme_produit(tuple(p["a"]), tuple(p["b"])))
    for p in paires
)
print(f"{en_francais(len(paires))} paires de formes comparées, avec PyTorch {torch.__version__}.")
print(f"Même règle que la page du site\u202f: {'oui' if regles and paires == fichier['paires'] else 'non'}")

if sur_la_carte:
    print(f"Ce carnet a tourné sur\u202f: la carte graphique, {torch.cuda.get_device_name(0)}.")
else:
    print("Ce carnet a tourné sur\u202f: le processeur.")  # \u202f est l’espace fine insécable

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/0.4-tenseurs sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/0.4-tenseurs).
- La page du chapitre, avec l’atelier des formes et les chronos du PC de La Forge IA : [chapitre 0.4](https://chipat-neko.github.io/forge-ia/partie-0/0-4/).
- Sur ton ordinateur, dans le dossier `projets/0.4-tenseurs` : `uv run --extra cpu python -m tenseurs formes "(2, 1, 3)" "(4, 1)"` applique les deux règles à deux formes de ton choix, et `uv run --extra cpu python -m tenseurs mesurer --appareil cpu` chronomètre ton processeur avec 1 fil, avec le réglage par défaut de PyTorch, puis avec tous ses fils. Avec une carte NVIDIA : `uv run --extra gpu python -m tenseurs mesurer --appareil cuda`.
- Le chapitre 0.5 assemble ces calculs en un premier réseau de neurones.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.